# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
import joblib

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [ ]:
csv_path = "../data/day-of-week-not-scaled.csv"
df = pd.read_csv(csv_path)
df_prev = pd.read_csv("../data/dayofweek.csv")
df['dayofweek'] = df_prev['dayofweek']

X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=0.2, random_state=21, stratify=y_train
)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [ ]:
svm = SVC(C=10, class_weight=None, gamma='auto', kernel='rbf',
          probability=True, random_state=21)
svm.fit(X_train, y_train)
y_pred = svm.predict(X_valid)

print(f"accuracy is {accuracy_score(y_valid, y_pred):.5f}")
print(f"precision is {precision_score(y_valid, y_pred, average='weighted'):.5f}")
print(f"recall is {recall_score(y_valid, y_pred, average='weighted'):.5f}")

In [ ]:
dt = DecisionTreeClassifier(class_weight='balanced', criterion='gini',
                            max_depth=21, random_state=21)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_valid)

print(f"accuracy is {accuracy_score(y_valid, y_pred):.5f}")
print(f"precision is {precision_score(y_valid, y_pred, average='weighted'):.5f}")
print(f"recall is {recall_score(y_valid, y_pred, average='weighted'):.5f}")

In [ ]:
rf = RandomForestClassifier(class_weight='balanced', criterion='entropy',
                            max_depth=24, n_estimators=100, random_state=21)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_valid)

print(f"accuracy is {accuracy_score(y_valid, y_pred):.5f}")
print(f"precision is {precision_score(y_valid, y_pred, average='weighted'):.5f}")
print(f"recall is {recall_score(y_valid, y_pred, average='weighted'):.5f}")

## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [ ]:
voting_clf = VotingClassifier(
    estimators=[('svm', svm), ('dt', dt), ('rf', rf)],
    voting='soft',
    weights=[4, 1, 4]
)
voting_clf.fit(X_train, y_train)

y_pred = voting_clf.predict(X_valid)
print(f"accuracy is {accuracy_score(y_valid, y_pred):.5f}")
print(f"precision is {precision_score(y_valid, y_pred, average='weighted'):.5f}")
print(f"recall is {recall_score(y_valid, y_pred, average='weighted'):.5f}")

In [ ]:
y_pred_test = voting_clf.predict(X_test)
print(f"accuracy is {accuracy_score(y_test, y_pred_test):.5f}")
print(f"precision is {precision_score(y_test, y_pred_test, average='weighted'):.5f}")
print(f"recall is {recall_score(y_test, y_pred_test, average='weighted'):.5f}")

## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [ ]:
svm_base = SVC(C=10, class_weight=None, gamma='auto', kernel='rbf',
               probability=True, random_state=21)

bagging = BaggingClassifier(
    base_estimator=svm_base,
    n_estimators=50,
    random_state=21
)
bagging.fit(X_train, y_train)

y_pred = bagging.predict(X_valid)
print(f"accuracy is {accuracy_score(y_valid, y_pred):.5f}")
print(f"precision is {precision_score(y_valid, y_pred, average='weighted'):.5f}")
print(f"recall is {recall_score(y_valid, y_pred, average='weighted'):.5f}")

In [ ]:
y_pred_test = bagging.predict(X_test)
print(f"accuracy is {accuracy_score(y_test, y_pred_test):.5f}")
print(f"precision is {precision_score(y_test, y_pred_test, average='weighted'):.5f}")
print(f"recall is {recall_score(y_test, y_pred_test, average='weighted'):.5f}")

## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [ ]:
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=21)

stacking = StackingClassifier(
    estimators=[('svm', svm), ('dt', dt), ('rf', rf)],
    final_estimator=LogisticRegression(solver='liblinear'),
    cv=cv,
    passthrough=True
)
stacking.fit(X_train, y_train)

y_pred = stacking.predict(X_valid)
print(f"accuracy is {accuracy_score(y_valid, y_pred):.5f}")
print(f"precision is {precision_score(y_valid, y_pred, average='weighted'):.5f}")
print(f"recall is {recall_score(y_valid, y_pred, average='weighted'):.5f}")

In [ ]:
y_pred_test = stacking.predict(X_test)
print(f"accuracy is {accuracy_score(y_test, y_pred_test):.5f}")
print(f"precision is {precision_score(y_test, y_pred_test, average='weighted'):.5f}")
print(f"recall is {recall_score(y_test, y_pred_test, average='weighted'):.5f}")

## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [ ]:
best_model = voting_clf

In [ ]:
y_pred_full = best_model.predict(X)

df_errors = df.copy()
df_errors['predicted'] = y_pred_full
df_errors['is_error'] = df_errors['dayofweek'] != df_errors['predicted']

In [ ]:
weekday_err = (df_errors.groupby('dayofweek')['is_error'].mean() * 100).round(2)
print('Error rate by weekday (%):')
print(weekday_err.sort_values(ascending=False))
print(f"\nWorst weekday: {weekday_err.idxmax()} → {weekday_err.max():.2f}%")

In [ ]:
if 'labname' in df_errors.columns:
    lab_err = (df_errors.groupby('labname')['is_error'].mean() * 100).round(2)
else:
    lab_cols = [col for col in df_errors.columns if col.startswith('labname_')]

    if lab_cols:
        df_errors['labname_restored'] = df_errors[lab_cols].idxmax(axis=1).str.replace('labname_', '')
        lab_err = (df_errors.groupby('labname_restored')['is_error'].mean() * 100).round(2)
    else:
        lab_err = None

if lab_err is not None:
    print('Error rate by labname (%):')
    print(lab_err.sort_values(ascending=False).head())
    print(f"\nWorst labname: {lab_err.idxmax()} → {lab_err.max():.2f}%")
else:
    print("\nlabname not available → analysis skipped")

In [ ]:
if 'uid' in df_errors.columns:
    user_err = (df_errors.groupby('uid')['is_error'].mean() * 100).round(2)
else:
    uid_cols = [col for col in df_errors.columns if col.startswith('uid_')]

    if uid_cols:
        df_errors['uid_restored'] = df_errors[uid_cols].idxmax(axis=1).str.replace('uid_', '')
        user_err = (df_errors.groupby('uid_restored')['is_error'].mean() * 100).round(2)
    else:
        user_err = None

if user_err is not None:
    print('Error rate by users (%):')
    print(user_err.sort_values(ascending=False).head())
    print(f"\nWorst user: {user_err.idxmax()} → {user_err.max():.2f}%")
else:
    print("\nuid not available → analysis skipped")

In [ ]:
joblib.dump(best_model, 'best_model.joblib')
print('Model saved')